In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze_table = "myadlasdevcalaog.bronze.customers"

df = spark.table(bronze_table)

display(df)

customer_id,customer_name,city,email,source_updated_date,ingestion_timestamp
101,Amit,Pune,amit@example.com,2026-10-01,2026-10-22T09:23:11.316Z
102,Priya,Mumbai,priya@example.com,2026-10-02,2026-10-22T09:23:11.316Z
103,Rahul,Nashik,null,2026-10-03,2026-10-22T09:23:11.316Z
104,Sneha,Pune,sneha@example.com,2026-10-04,2026-10-22T09:23:11.316Z
105,Vijay,Nagpur,vijay@example.com,2026-10-05,2026-10-22T09:23:11.316Z


In [0]:
# Remove records with invalid customer IDs or names
df_valid = df.filter(
    F.col("customer_id").isNotNull() &
    (F.col("customer_id") > 0) &
    F.col("customer_name").isNotNull() &
    (F.trim(F.col("customer_name")) != "")
)

# Keep the latest record for each customer
window_spec = Window.partitionBy("customer_id").orderBy(
    F.col("source_updated_date").desc_nulls_last(),
    F.col("ingestion_timestamp").desc()
)

df_deduplicated = (
    df_valid
    .withColumn("row_num", F.row_number().over(window_spec))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

# Handle missing or blank email addresses
df_silver = df_deduplicated.withColumn(
    "email",
    F.when(
        F.col("email").isNull() | (F.trim(F.col("email")) == ""),
        F.lit("unknown")
    ).otherwise(F.trim(F.col("email")))
)

display(df_silver)

customer_id,customer_name,city,email,source_updated_date,ingestion_timestamp
101,Amit,Pune,amit@example.com,2026-10-01,2026-10-22T09:23:11.316Z
102,Priya,Mumbai,priya@example.com,2026-10-02,2026-10-22T09:23:11.316Z
103,Rahul,Nashik,unknown,2026-10-03,2026-10-22T09:23:11.316Z
104,Sneha,Pune,sneha@example.com,2026-10-04,2026-10-22T09:23:11.316Z
105,Vijay,Nagpur,vijay@example.com,2026-10-05,2026-10-22T09:23:11.316Z


In [0]:
%sql
SELECT
    COUNT(*) AS total_customers,
    COUNT(DISTINCT customer_id) AS unique_customers,
    SUM(CASE WHEN email = 'unknown' THEN 1 ELSE 0 END)
        AS missing_email_count
FROM myadlasdevcalaog.silver.customers;

total_customers,unique_customers,missing_email_count
5,5,1


In [0]:
%sql
SELECT customer_id, customer_name, city, source_updated_date
FROM myadlasdevcalaog.silver.customers
ORDER BY customer_id;

customer_id,customer_name,city,source_updated_date
101,Amit,Pune,2026-10-01
102,Priya,Mumbai,2026-10-02
103,Rahul,Nashik,2026-10-03
104,Sneha,Pune,2026-10-04
105,Vijay,Nagpur,2026-10-05


In [0]:
%sql
SELECT COUNT(*) AS total_customers
FROM myadlasdevcalaog.silver.customers;

total_customers
5


In [0]:

from delta.tables import DeltaTable

silver_table = "myadlasdevcalaog.silver.customers"

if spark.catalog.tableExists(silver_table):
    delta_target = DeltaTable.forName(spark, silver_table)

    (
        delta_target.alias("target")
        .merge(
            df_silver.alias("source"),
            "target.customer_id = source.customer_id"
        )
        .whenMatchedUpdateAll(
            condition="""
                source.source_updated_date >= target.source_updated_date
            """
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    print("Silver table updated using MERGE")

else:
    (
        df_silver.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(silver_table)
    )

    print("Silver table created for initial load")


Silver table updated using MERGE


In [0]:
%sql
SELECT customer_id, customer_name, city, source_updated_date
FROM myadlasdevcalaog.silver.customers
WHERE customer_id IN (101, 106)
ORDER BY customer_id;

customer_id,customer_name,city,source_updated_date
101,Amit,Pune,2026-10-01


In [0]:
print("Silver source rows:", df_silver.count())

display(
    df_silver.select(
        "customer_id",
        "customer_name",
        "city",
        "source_updated_date"
    ).orderBy("customer_id")
)

Silver source rows: 5


customer_id,customer_name,city,source_updated_date
101,Amit,Pune,2026-10-01
102,Priya,Mumbai,2026-10-02
103,Rahul,Nashik,2026-10-03
104,Sneha,Pune,2026-10-04
105,Vijay,Nagpur,2026-10-05
